# Music Annotation Project V8.6 - Notebook 45
## Residual Mask Readiness and Oracle Audit

**Runtime: CPU.** Development-only. This notebook constructs and locks M0-M3 without clean truth, then loads clean truth to create diagnostic oracle M4 and score residual-mask readiness. It does not train a restorer and does not authorise production use.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib

In [ ]:
from pathlib import Path
from datetime import datetime,timezone
import json,hashlib,math,shutil,sys,platform,zipfile
import cv2,numpy as np,pandas as pd,matplotlib.pyplot as plt
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V84_ROOT=ROOT/'observed_print_classifier_v8_4'
OUTROOT=ROOT/'observed_print_classifier_v8_6_notebook45'
SOURCE_V84_RUN_OVERRIDE=None
SEED=45001; PRIMARY_COLLAR_RADIUS=2; COLLAR_SENSITIVITY=[1,2,3]; MIN_REGION_AREA=2; REVIEW_MARGIN=120
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v8_6_notebook45'
OUT=OUTROOT/RUN; TABLES=OUT/'tables'; MASKS=OUT/'masks'; REVIEW=OUT/'visual_audit'; AUDIT=OUT/'audit'; UPLOAD=OUT/'upload'
for p in [OUT,TABLES,MASKS,REVIEW,AUDIT,UPLOAD]:p.mkdir(parents=True,exist_ok=False)
PRE={'version':'8.6-notebook45-development','development_only':True,'promotion_authority':False,'mask_arms':['M0','M1','M2','M3','M4'],'primary_collar_radius':PRIMARY_COLLAR_RADIUS,'collar_sensitivity':COLLAR_SENSITIVITY,'truth_roles':{'M0_M3':'prohibited','M4_and_scoring':'post_lock_only'},'seed':SEED}
(OUT/'01_pre_registration.json').write_text(json.dumps(PRE,indent=2))

In [ ]:
def sha(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def ah(a):
 a=np.ascontiguousarray(a);return hashlib.sha256(str(a.dtype).encode()+str(a.shape).encode()+a.tobytes()).hexdigest()
def oh(x):return hashlib.sha256(json.dumps(x,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()
def latest(root,req):
 hits=[p for p in Path(root).rglob('*') if p.is_dir() and all((p/r).exists() for r in req)] if Path(root).exists() else []
 return max(hits,key=lambda p:p.stat().st_mtime) if hits else None
REQ=['14_experiment_manifest.json','15_v8_4_verdict.json','candidate_lock.json','tables/04_oof_safety_and_value_predictions.parquet','tables/02_candidate_group_pixels.parquet']
SOURCE=Path(SOURCE_V84_RUN_OVERRIDE) if SOURCE_V84_RUN_OVERRIDE else latest(V84_ROOT,REQ)
if SOURCE is None:raise RuntimeError('No complete Notebook 41 V8.4 run found.')
manifest=json.loads((SOURCE/'14_experiment_manifest.json').read_text()); verdict=json.loads((SOURCE/'15_v8_4_verdict.json').read_text())
oof=pd.read_parquet(SOURCE/'tables/04_oof_safety_and_value_predictions.parquet'); pix=pd.read_parquet(SOURCE/'tables/02_candidate_group_pixels.parquet')
if 'arm' in oof:oof=oof[oof.arm=='multiscale_33_97'].copy()
required={'group_id','page','selected'}
if required-set(oof):raise RuntimeError(f'OOF predictions missing {required-set(oof)}')
if not {'group_id','page','y','x'}.issubset(pix):raise RuntimeError('Candidate pixels table schema incomplete.')
source_v82=Path(manifest['source_record']['source']); v75=Path(manifest['source_record']['v75'])
status=pd.read_csv(v75/'tables/cache_status.csv'); cache={str(r.page):Path(r.cache) for _,r in status.iterrows()}
record={'source_v84':str(SOURCE),'manifest_sha256':sha(SOURCE/'14_experiment_manifest.json'),'verdict_sha256':sha(SOURCE/'15_v8_4_verdict.json'),'candidate_lock_sha256':sha(SOURCE/'candidate_lock.json'),'source_v82':str(source_v82),'v75':str(v75)}
(OUT/'source_record.json').write_text(json.dumps(record,indent=2));print(json.dumps(record,indent=2))

## Build M0-M3 without loading clean truth

In [ ]:
req_arrays=['mask','g80_ambiguous','g80_preserve']
regions=[]; page_data={}; semantic={}
for page in sorted(oof.page.unique()):
 if page not in cache:raise RuntimeError(f'{page}: no V7.5 cache')
 with np.load(cache[page]) as z:
  miss=[k for k in req_arrays if k not in z.files]
  if miss:raise RuntimeError(f'{page}: missing {miss}')
  auth=z['mask']>0; m0=z['g80_ambiguous']>0; g78=z['g80_preserve']>0
 obs=cv2.imread(str(source_v82/'whole_pages'/page/'observed.png'),0); base=cv2.imread(str(source_v82/'whole_pages'/page/'v7_8.png'),0)
 if obs is None or base is None or obs.shape!=auth.shape:raise RuntimeError(f'{page}: page identity failure')
 p82=pd.read_parquet(source_v82/'tables/all_eligible_oof_predictions.parquet') if page==sorted(oof.page.unique())[0] else p82
 pg=p82[p82.page==page]; strict=np.zeros(auth.shape,bool)
 if 'decision' in pg: hit=pg.decision.astype(str).eq('PRINT').to_numpy()
 else: hit=pg.p_print.to_numpy(float)>=.995
 strict[pg.y.to_numpy(int)[hit],pg.x.to_numpy(int)[hit]]=True
 selected=set(oof.loc[(oof.page==page)&oof.selected.astype(bool),'group_id'].astype(str)); v84=np.zeros(auth.shape,bool)
 q=pix[(pix.page==page)&pix.group_id.astype(str).isin(selected)];v84[q.y.to_numpy(int),q.x.to_numpy(int)]=True
 protected=g78|strict|v84; m1=m0&~strict; m2=m1&~v84
 uncertain_groups=set(oof.loc[(oof.page==page)&~oof.selected.astype(bool),'group_id'].astype(str)); uq=pix[(pix.page==page)&pix.group_id.astype(str).isin(uncertain_groups)]; uncertainty=np.zeros(auth.shape,bool);uncertainty[uq.y.to_numpy(int),uq.x.to_numpy(int)]=True
 collar_source=protected|uncertainty; k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*PRIMARY_COLLAR_RADIUS+1,2*PRIMARY_COLLAR_RADIUS+1)); collar=(cv2.dilate(collar_source.astype(np.uint8),k)>0)&auth&~protected
 m3=m2&~collar
 page_data[page]={'obs':obs,'base':base,'auth':auth,'protected':protected,'M0':m0,'M1':m1,'M2':m2,'M3':m3,'collar':collar,'strict':strict,'v84':v84}
 n,lab,stats,_=cv2.connectedComponentsWithStats((m0|m1|m2|m3).astype(np.uint8),8)
 for cid in range(1,n):
  comp=lab==cid
  if comp.sum()<MIN_REGION_AREA:continue
  ys,xs=np.where(comp); rid=f'{page}:r{cid:05d}'
  regions.append({'region_id':rid,'page':page,'source_group':str(oof.loc[oof.page==page,'source_group'].iloc[0]) if 'source_group' in oof else 'UNKNOWN','component_id':cid,'x0':int(xs.min()),'y0':int(ys.min()),'x1':int(xs.max()+1),'y1':int(ys.max()+1),'union_pixels':int(comp.sum())})
 semantic[page]={k:ah(v.astype(np.uint8)) for k,v in page_data[page].items() if isinstance(v,np.ndarray)}
regions=pd.DataFrame(regions).sort_values(['page','component_id']);regions.to_parquet(TABLES/'region_manifest.parquet',index=False)
lock={'pre_registration':PRE,'source_record':record,'regions_sha256':sha(TABLES/'region_manifest.parquet'),'semantic_M0_M3':semantic};(OUT/'mask_lock.json').write_text(json.dumps({'lock_id':oh(lock),'payload':lock},indent=2))
print('LOCKED M0-M3 regions:',len(regions))

## Post-lock: construct M4 and calculate readiness metrics

In [ ]:
clean_map={p.stem:p for p in (ROOT/'source'/'clean').iterdir() if p.suffix.lower() in {'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}}
def printed_ink(im):return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def arm_features(obs,mask,protected):
 n,lab,stats,_=cv2.connectedComponentsWithStats(mask.astype(np.uint8),8);depth=cv2.distanceTransform(mask.astype(np.uint8),cv2.DIST_L2,5);edge=cv2.Canny(obs,70,170)>0;boundary=(cv2.dilate(mask.astype(np.uint8),np.ones((3,3),np.uint8))>0)&~mask
 return {'residual_pixels':int(mask.sum()),'residual_components':int(max(n-1,0)),'maximum_depth':float(depth.max()),'boundary_pixels':int(boundary.sum()),'protected_boundary_contact':float((boundary&cv2.dilate(protected.astype(np.uint8),np.ones((3,3),np.uint8)).astype(bool)).sum()/max(boundary.sum(),1)),'boundary_edge_rate':float(edge[boundary].mean()) if boundary.any() else 0.0}
rows=[]; oracle_rows=[]
for _,r in regions.iterrows():
 page=r.page;d=page_data[page];clean=cv2.imread(str(clean_map[page]),0);ci=printed_ink(clean);obs_ink=printed_ink(d['obs']);truth_print=ci&obs_ink&d['auth']; truth_annotation=obs_ink&~ci&d['auth']; unsupported=ci&d['auth']&~obs_ink
 # M4 reconstructs only genuine clean ink not available as observed bytes; annotation-only is removable, not reconstructable.
 M4=unsupported.copy();d['M4']=M4
 y0,y1,x0,x1=int(r.y0),int(r.y1),int(r.x0),int(r.x1);region=np.zeros_like(d['auth']);region[y0:y1,x0:x1]=True
 for arm in ['M0','M1','M2','M3','M4']:
  m=d[arm]&region; f=arm_features(d['obs'],m,d['protected']);f.update({'region_id':r.region_id,'page':page,'source_group':r.source_group,'arm':arm,'genuine_print_in_residual':int((m&truth_print).sum()),'annotation_in_residual':int((m&truth_annotation).sum()),'unsupported_print_target':int((m&unsupported).sum()),'false_annotation_protected':int((d['protected']&truth_annotation&region).sum()),'genuine_print_protected':int((d['protected']&truth_print&region).sum())});rows.append(f)
  rd=MASKS/page/r.region_id.replace(':','_');rd.mkdir(parents=True,exist_ok=True);cv2.imwrite(str(rd/f'{arm}.png'),m.astype(np.uint8)*255)
metrics=pd.DataFrame(rows);metrics.to_csv(TABLES/'mask_readiness_metrics.csv',index=False)
wide=metrics.pivot(index=['region_id','page','source_group'],columns='arm',values=['genuine_print_in_residual','residual_pixels','protected_boundary_contact']).reset_index()
wide.columns=['_'.join([str(x) for x in c if str(x)]) for c in wide.columns]
for metric in ['genuine_print_in_residual','residual_pixels','protected_boundary_contact']:
 den=wide[f'{metric}_M4']-wide[f'{metric}_M0']; num=wide[f'{metric}_M3']-wide[f'{metric}_M0'];wide[f'{metric}_oracle_normalised']=np.where(np.abs(den)>1e-9,num/den,np.nan)
wide.to_csv(TABLES/'oracle_gap_summary.csv',index=False)
print(metrics.groupby('arm').agg(regions=('region_id','nunique'),residual_pixels=('residual_pixels','sum'),genuine_print_in_residual=('genuine_print_in_residual','sum'),annotation_in_residual=('annotation_in_residual','sum'),boundary_contact=('protected_boundary_contact','mean')))

In [ ]:
# Collar sensitivity is mask-only and uses the already locked observable sources.
sens=[]
for page,d in page_data.items():
 src=d['protected']|d['collar']
 for radius in COLLAR_SENSITIVITY:
  k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*radius+1,2*radius+1)); col=(cv2.dilate(src.astype(np.uint8),k)>0)&d['auth']&~d['protected']; m=d['M2']&~col
  sens.append({'page':page,'radius':radius,'residual_pixels':int(m.sum()),'collar_pixels':int(col.sum())})
pd.DataFrame(sens).to_csv(TABLES/'collar_sensitivity.csv',index=False)

In [ ]:
# Visual review panels: observed, M0, M2, M3, M4; clean is intentionally diagnostic last.
for _,r in regions.head(200).iterrows():
 page=r.page;d=page_data[page];clean=cv2.imread(str(clean_map[page]),0);m=REVIEW_MARGIN;x0=max(0,int(r.x0)-m);x1=min(d['obs'].shape[1],int(r.x1)+m);y0=max(0,int(r.y0)-m);y1=min(d['obs'].shape[0],int(r.y1)+m)
 panels=[d['obs'],d['M0'].astype(np.uint8)*255,d['M2'].astype(np.uint8)*255,d['M3'].astype(np.uint8)*255,d['M4'].astype(np.uint8)*255,clean];titles=['observed','M0','M2','M3','M4 oracle','clean diagnostic']
 fig,axes=plt.subplots(1,6,figsize=(18,3.2))
 for ax,im,t in zip(axes,panels,titles):ax.imshow(im[y0:y1,x0:x1],cmap='gray',vmin=0,vmax=255);ax.set_title(t);ax.axis('off')
 plt.tight_layout();plt.savefig(REVIEW/f'{r.region_id.replace(":","_")}.png',dpi=150,bbox_inches='tight');plt.close(fig)

In [ ]:
agg=metrics.groupby('arm',as_index=False).agg(regions=('region_id','nunique'),residual_pixels=('residual_pixels','sum'),genuine_print_in_residual=('genuine_print_in_residual','sum'),annotation_in_residual=('annotation_in_residual','sum'),protected_boundary_contact=('protected_boundary_contact','mean'))
a=agg.set_index('arm'); valid=bool((a.loc['M3','genuine_print_in_residual']<=a.loc['M2','genuine_print_in_residual']) and (a.loc['M3','protected_boundary_contact']>=a.loc['M0','protected_boundary_contact']))
verdict={'version':'8.6-notebook45','valid_run':True,'ready_for_crossover':valid,'interpretation':'READY_FOR_CROSSOVER' if valid else 'MASK_RESEARCH_REQUIRED','mask_lock_id':json.loads((OUT/'mask_lock.json').read_text())['lock_id'],'primary_collar_radius':PRIMARY_COLLAR_RADIUS}
(OUT/'notebook45_verdict.json').write_text(json.dumps(verdict,indent=2));agg.to_csv(TABLES/'mask_arm_summary.csv',index=False)

In [ ]:
# Standard curated upload package.
artifacts=['01_pre_registration.json','source_record.json','mask_lock.json','notebook45_verdict.json','tables/region_manifest.parquet','tables/mask_readiness_metrics.csv','tables/mask_arm_summary.csv','tables/oracle_gap_summary.csv','tables/collar_sensitivity.csv']
rows=[]
for rel in artifacts:
 p=OUT/rel
 if not p.exists():raise RuntimeError(f'Missing upload artefact {rel}')
 dst=UPLOAD/rel;dst.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,dst);rows.append({'relative_path':rel,'bytes':p.stat().st_size,'sha256':sha(p)})
pd.DataFrame(rows).to_csv(UPLOAD/'upload_manifest.csv',index=False);(UPLOAD/'README.txt').write_text('V8.6 Notebook 45 curated review package. Development evidence only.\n');(UPLOAD/'upload_file_list.txt').write_text('\n'.join(x['relative_path'] for x in rows))
zip_path=UPLOAD/'notebook45_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
 for p in UPLOAD.rglob('*'):
  if p.is_file() and p!=zip_path:z.write(p,p.relative_to(UPLOAD))
print('NOTEBOOK 45 COMPLETE:',OUT)